In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

train = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")
test = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")

X = train.drop(columns=['label']).values.astype(np.float32) / 255.0
y = train['label'].values.reshape(-1, 1).astype(int)

encoder = OneHotEncoder(sparse_output=False)
y_oh = encoder.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(X, y_oh, test_size=0.1, random_state=42)


def init_params(input_size=784, hidden_size=256, output_size=10):
    np.random.seed(42)
    W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2. / input_size)
    b1 = np.zeros((1, hidden_size))
    W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2. / hidden_size)
    b2 = np.zeros((1, output_size))
    return W1, b1, W2, b2


# Activation functions
def relu(Z): return np.maximum(0, Z)
def relu_deriv(Z): return (Z > 0).astype(float)
def softmax(Z):
    Z_shift = Z - np.max(Z, axis=1, keepdims=True)
    expZ = np.exp(Z_shift)
    return expZ / np.sum(expZ, axis=1, keepdims=True)

def forward(X, W1, b1, W2, b2):
    Z1 = X @ W1 + b1
    A1 = relu(Z1)
    Z2 = A1 @ W2 + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

def compute_loss(y_true, y_pred):
    m = y_true.shape[0]
    return -np.sum(y_true * np.log(y_pred + 1e-8)) / m

def backward(X, y_true, Z1, A1, A2, W2):
    m = X.shape[0]
    dZ2 = A2 - y_true
    dW2 = (A1.T @ dZ2) / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * relu_deriv(Z1)
    dW1 = (X.T @ dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m
    return dW1, db1, dW2, db2

def accuracy(X, y_true, W1, b1, W2, b2):
    _, _, _, A2 = forward(X, W1, b1, W2, b2)
    preds = np.argmax(A2, axis=1)
    true_labels = np.argmax(y_true, axis=1)
    return np.mean(preds == true_labels)

def train_adam(X, y, X_val, y_val, epochs=15, batch_size=128, lr=0.001):
    W1, b1, W2, b2 = init_params()
    beta1, beta2, eps = 0.9, 0.999, 1e-8
    mW1, vW1 = np.zeros_like(W1), np.zeros_like(W1)
    mB1, vB1 = np.zeros_like(b1), np.zeros_like(b1)
    mW2, vW2 = np.zeros_like(W2), np.zeros_like(W2)
    mB2, vB2 = np.zeros_like(b2), np.zeros_like(b2)
    t = 0

    for epoch in range(epochs):
        idx = np.random.permutation(X.shape[0])
        X, y = X[idx], y[idx]

        for i in range(0, X.shape[0], batch_size):
            X_batch = X[i:i+batch_size]
            y_batch = y[i:i+batch_size]

            Z1, A1, Z2, A2 = forward(X_batch, W1, b1, W2, b2)
            dW1, db1, dW2, db2 = backward(X_batch, y_batch, Z1, A1, A2, W2)

            t += 1
            for param, grad, m, v in zip(
                [W1, b1, W2, b2],
                [dW1, db1, dW2, db2],
                [mW1, mB1, mW2, mB2],
                [vW1, vB1, vW2, vB2],
            ):
                m[:] = beta1 * m + (1 - beta1) * grad
                v[:] = beta2 * v + (1 - beta2) * (grad ** 2)
                m_corr = m / (1 - beta1 ** t)
                v_corr = v / (1 - beta2 ** t)
                param -= lr * m_corr / (np.sqrt(v_corr) + eps)

        val_acc = accuracy(X_val, y_val, W1, b1, W2, b2)
        print(f"Epoch {epoch+1}/{epochs} | Val Acc: {val_acc*100:.2f}%")

    return W1, b1, W2, b2

W1, b1, W2, b2 = train_adam(X_train, y_train, X_val, y_val, epochs=27, lr=0.001)

final_acc = accuracy(X_val, y_val, W1, b1, W2, b2)
print(f"\n Final validation accuracy: {final_acc*100:.2f}%")
print(f"\n")

m = 5  # small batch size
X_demo = np.random.randn(m, 784)
W_demo = np.random.randn(784, 10)
b_demo = np.random.randn(10)

Z = X_demo @ W_demo + b_demo  # shape (m,10)
A = np.exp(Z) / np.sum(np.exp(Z), axis=1, keepdims=True)  # softmax

y_true = np.zeros((m, 10))
y_true[np.arange(m), np.random.randint(0, 10, size=m)] = 1


loss = -np.sum(y_true * np.log(A + 1e-8)) / m

dZ = A - y_true
dW = X_demo.T @ dZ / m
db = np.sum(dZ, axis=0) / m

lr = 0.01
W_demo -= lr * dW
b_demo -= lr * db

print(f"Loss: {loss:.4f}")
print("Updated weights shape:", W_demo.shape)
print("Updated biases shape:", b_demo.shape)
